# 03. PPG-only alternative models and Optuna: code-inspection notebook

이 노트북은 대안모델 실험만 분리한 파일입니다. CLI를 호출하지 않고,
`tuning.py`와 `modeling.py`의 핵심 구현을 기능 단위 셀로 펼쳐서 직접 실행합니다.

권장 순서: `02_lightgbm_baselines.ipynb`에서 feature table을 만든 뒤 이 노트북을 실행합니다.

## 0. Notebook runtime configuration

In [ ]:
from __future__ import annotations

import hashlib
import json
import platform
import time
import warnings
from collections import Counter
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Literal, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
from IPython.display import Markdown, display

def find_repo_root(start: Path | None = None) -> Path:
    """Find repo root from either JupyterLab cwd or notebook-directory cwd."""
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "llamac_research").exists():
            return candidate
    raise RuntimeError(f"Could not locate llamac_research repo root from {start}")

ROOT = find_repo_root()
PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "artifacts" / "results"
OPTUNA_BASE = ROOT / "artifacts" / "optuna" / "ppg_reported_grouped"
OPTUNA_RICH = ROOT / "artifacts" / "optuna" / "ppg_rich_reported_grouped"
FEATURE_PPG = PROCESSED / "features_ppg.parquet"
FEATURE_PPG_RICH = PROCESSED / "features_ppg_rich.parquet"

SEED = 42
FOLDS = 5
DEVICE = "auto"  # auto = try CUDA/GPU first when supported, then fall back to CPU.
BASE_TRIALS = 20
SVC_TRIALS = 10
RICH_TRIALS = 20
RERUN_TUNING = False

for path in [RESULTS, OPTUNA_BASE, OPTUNA_RICH]:
    path.mkdir(parents=True, exist_ok=True)

## 1. Label and metric helpers

Source: `labels.py`, `metrics.py`. 대안모델도 같은 label/metric contract를 사용합니다.

In [ ]:
EMOTION_ID_TO_LABEL: dict[int, str] = {
    1: "neutral",
    2: "fun",
    3: "sadness",
    4: "anger",
    5: "fear",
}
EMOTION_LABELS: list[str] = [EMOTION_ID_TO_LABEL[i] for i in sorted(EMOTION_ID_TO_LABEL)]
EMOTION_IDS: list[int] = sorted(EMOTION_ID_TO_LABEL)

ANSWER_COLUMNS: list[str] = [
    "SubjectID",
    "Trial",
    "NoVideo",
    "Valence",
    "Arousal",
    "Dominance",
    "Liking",
    "EmotType",
    "EmotStr",
    "Seen",
]

TARGET_COLUMNS: list[str] = ["IntendedType", "ReportedType"]
SELF_REPORT_COLUMNS: list[str] = [
    "NoVideo",
    "Valence",
    "Arousal",
    "Dominance",
    "Liking",
    "EmotType",
    "EmotStr",
    "Seen",
    *TARGET_COLUMNS,
]

OFFICIAL_EXCLUDE_SUBJECTS: tuple[int, ...] = (32, 37, 40, 47, 54, 55, 56, 70, 99)
OFFICIAL_EXCLUDE_TRIALS: dict[int, tuple[int, ...]] = {
    7: (27, 28, 36),
    19: (18,),
    59: (35, 13),
    111: (1,),
    20: (26,),
    89: (13,),
    105: (38, 50),
    107: (38,),
}

@dataclass(frozen=True)
class TargetSpec:
    """Resolved target-column metadata."""

    name: str
    column: str
    labels: list[int]
    label_names: list[str]

def resolve_target(name: str) -> TargetSpec:
    """Resolve a CLI target name to a dataframe column."""
    normalized = name.lower().strip().replace("-", "_")
    if normalized in {"reported", "reported_type", "self_report", "emottype", "emotion"}:
        return TargetSpec("reported", "ReportedType", EMOTION_IDS, EMOTION_LABELS)
    if normalized in {"intended", "intended_type", "targeted", "novideo"}:
        return TargetSpec("intended", "IntendedType", EMOTION_IDS, EMOTION_LABELS)
    raise ValueError(f"Unsupported target {name!r}; use 'reported' or 'intended'.")

def filter_official_valid_trials(frame: pl.DataFrame) -> pl.DataFrame:
    """Apply the official notebook's subject/trial exclusions."""
    required = {"SubjectID", "Trial"}
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"Missing columns for official filtering: {sorted(missing)}")

    out = frame.with_columns(
        pl.col("SubjectID").cast(pl.Int64, strict=False).alias("__subject_int"),
        pl.col("Trial").cast(pl.Int64, strict=False).alias("__trial_int"),
    )
    keep = ~pl.col("__subject_int").is_in(list(OFFICIAL_EXCLUDE_SUBJECTS))
    for sid, trials in OFFICIAL_EXCLUDE_TRIALS.items():
        keep = keep & ~((pl.col("__subject_int") == sid) & pl.col("__trial_int").is_in(list(trials)))
    return out.filter(keep).drop(["__subject_int", "__trial_int"])

@dataclass(frozen=True)
class ClassificationMetrics:
    """Serializable metric bundle."""

    top1_accuracy: float
    top2_accuracy: float
    top3_accuracy: float
    macro_f1: float
    weighted_f1: float
    balanced_accuracy: float
    cohen_kappa: float
    confusion_matrix: list[list[int]]
    per_class: dict[str, dict[str, float | int]]

    def to_dict(self) -> dict[str, Any]:
        return {
            "top1_accuracy": self.top1_accuracy,
            "top2_accuracy": self.top2_accuracy,
            "top3_accuracy": self.top3_accuracy,
            "macro_f1": self.macro_f1,
            "weighted_f1": self.weighted_f1,
            "balanced_accuracy": self.balanced_accuracy,
            "cohen_kappa": self.cohen_kappa,
            "confusion_matrix": self.confusion_matrix,
            "per_class": self.per_class,
        }

def _top_k_accuracy(y_true: np.ndarray, y_score: np.ndarray, labels: Sequence[int], k: int) -> float:
    if y_true.size == 0:
        return float("nan")
    label_arr = np.asarray(labels)
    k = min(k, y_score.shape[1])
    top_idx = np.argpartition(y_score, kth=y_score.shape[1] - k, axis=1)[:, -k:]
    top_labels = label_arr[top_idx]
    return float(np.mean([yt in row for yt, row in zip(y_true, top_labels, strict=False)]))

def compute_classification_metrics(
    y_true: Sequence[int],
    y_pred: Sequence[int],
    y_score: Sequence[Sequence[float]] | np.ndarray | None = None,
    *,
    labels: Sequence[int] = EMOTION_IDS,
    label_names: Sequence[str] = EMOTION_LABELS,
) -> ClassificationMetrics:
    """Compute the required LLaMAC classification metrics."""
    from sklearn.metrics import (
        balanced_accuracy_score,
        cohen_kappa_score,
        confusion_matrix,
        f1_score,
        precision_recall_fscore_support,
    )

    y_true_arr = np.asarray(y_true, dtype=int)
    y_pred_arr = np.asarray(y_pred, dtype=int)
    label_list = list(labels)
    if y_score is None:
        score = np.zeros((y_pred_arr.size, len(label_list)), dtype=float)
        index = {label: idx for idx, label in enumerate(label_list)}
        for row, pred in enumerate(y_pred_arr):
            if pred in index:
                score[row, index[pred]] = 1.0
    else:
        score = np.asarray(y_score, dtype=float)
    top1 = float(np.mean(y_true_arr == y_pred_arr)) if y_true_arr.size else float("nan")
    top2 = _top_k_accuracy(y_true_arr, score, label_list, 2)
    top3 = _top_k_accuracy(y_true_arr, score, label_list, 3)
    macro = float(f1_score(y_true_arr, y_pred_arr, labels=label_list, average="macro", zero_division=0))
    weighted = float(f1_score(y_true_arr, y_pred_arr, labels=label_list, average="weighted", zero_division=0))
    balanced = float(balanced_accuracy_score(y_true_arr, y_pred_arr))
    kappa = float(cohen_kappa_score(y_true_arr, y_pred_arr, labels=label_list))
    cm = confusion_matrix(y_true_arr, y_pred_arr, labels=label_list)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true_arr,
        y_pred_arr,
        labels=label_list,
        zero_division=0,
    )
    per_class = {
        name: {
            "label_id": int(label),
            "precision": float(p),
            "recall": float(r),
            "f1": float(f),
            "support": int(s),
        }
        for label, name, p, r, f, s in zip(label_list, label_names, precision, recall, f1, support, strict=True)
    }
    return ClassificationMetrics(
        top1_accuracy=top1,
        top2_accuracy=top2,
        top3_accuracy=top3,
        macro_f1=macro,
        weighted_f1=weighted,
        balanced_accuracy=balanced,
        cohen_kappa=kappa,
        confusion_matrix=cm.astype(int).tolist(),
        per_class=per_class,
    )

def align_proba_columns(model_classes: Sequence[int], proba: np.ndarray, labels: Sequence[int] = EMOTION_IDS) -> np.ndarray:
    """Align estimator predict_proba columns to the canonical label order."""
    classes = [int(c) for c in model_classes]
    label_list = list(labels)
    aligned = np.zeros((proba.shape[0], len(label_list)), dtype=float)
    for src_idx, label in enumerate(classes):
        if label in label_list:
            aligned[:, label_list.index(label)] = proba[:, src_idx]
    row_sum = aligned.sum(axis=1, keepdims=True)
    missing = row_sum.squeeze() == 0
    if np.any(missing):
        aligned[missing, :] = 1.0 / len(label_list)
    else:
        aligned = aligned / row_sum
    return aligned

def metrics_summary_line(metrics: dict[str, Any]) -> str:
    """Compact human-readable metric line for CLI logs."""
    return (
        f"top1={metrics['top1_accuracy']:.4f} top2={metrics['top2_accuracy']:.4f} "
        f"top3={metrics['top3_accuracy']:.4f} macro_f1={metrics['macro_f1']:.4f} "
        f"balanced_acc={metrics['balanced_accuracy']:.4f} kappa={metrics['cohen_kappa']:.4f}"
    )

## 2. Read prepared feature tables and inspect feature variants

이 노트북은 feature extraction을 다시 숨기지 않습니다. feature 생성 코드는 baseline 노트북에 펼쳐져 있고,
여기서는 대안모델 실험을 위해 parquet table을 직접 읽습니다.

`ppg`와 `ppg_rich`는 둘 다 `band_*.csv`의 `PPG` 신호만 사용하는 PPG-only table입니다. `ppg_rich`는 GSR, SKT, EEG, respiration을 추가한 multimodal table이 아니라, 기본 PPG 통계/HRV feature에 PPG 파생 feature를 더한 확장판입니다.

`ppg_rich`가 추가하는 계열은 1차/2차 차분 통계(`Band_PPG_diff*`), peak morphology와 peak rate(`Band_PPG_peak*`), Welch PSD 기반 spectral power/entropy/dominant frequency(`Band_PPG_*_power_*`, `Band_PPG_spec_entropy`, `Band_PPG_dominant_hz`), 그리고 trial을 4구간으로 나눈 temporal segment mean/std/slope(`Band_PPG_seg*`)입니다.

In [ ]:
def read_feature_table(path: str | Path) -> pl.DataFrame:
    p = Path(path)
    if p.suffix.lower() == ".parquet":
        return pl.read_parquet(p)
    return pl.read_csv(p, infer_schema_length=1024, ignore_errors=True)

if not FEATURE_PPG.exists() or not FEATURE_PPG_RICH.exists():
    raise FileNotFoundError("Run 02_lightgbm_baselines.ipynb feature-building cells first.")

ppg = read_feature_table(FEATURE_PPG)
ppg_rich = read_feature_table(FEATURE_PPG_RICH)
base_cols = set(ppg.columns)
rich_cols = set(ppg_rich.columns)
rich_added = sorted(c for c in rich_cols - base_cols if c.startswith("Band_PPG"))

def rich_feature_family(name: str) -> str:
    if "_diff" in name:
        return "derivative_statistics"
    if "_peak" in name:
        return "peak_morphology"
    if "_power_" in name or name.endswith("_dominant_hz") or name.endswith("_spec_entropy"):
        return "spectral_welch_psd"
    if "_seg" in name:
        return "temporal_segments"
    return "other_ppg"

rich_family_counts = Counter(rich_feature_family(c) for c in rich_added)

display(pd.DataFrame([
    {"variant": "ppg", "rows": ppg.height, "columns": ppg.width},
    {"variant": "ppg_rich", "rows": ppg_rich.height, "columns": ppg_rich.width, "rich_only_ppg_features": len(rich_added)},
]))
display(pd.DataFrame([
    {"family": family, "added_features": count}
    for family, count in sorted(rich_family_counts.items())
]))
display(pd.DataFrame({"rich_only_feature_example": rich_added[:40]}))

## 3. Modeling harness: feature matrix, preprocessing, splitters

Source: `modeling.py`. 여기에 각 fold별 median imputation/drop-column 처리와 grouped split이 있습니다.

In [ ]:
__version__ = "0.1.0"
FeatureSet = Literal["all", "ppg"]
SplitStrategy = Literal["grouped", "stratified"]
ModelName = Literal[
    "lightgbm",
    "logistic_regression",
    "svc_rbf",
    "random_forest",
    "extra_trees",
    "hist_gradient_boosting",
    "xgboost",
]
MODEL_NAMES: tuple[str, ...] = (
    "lightgbm",
    "logistic_regression",
    "svc_rbf",
    "random_forest",
    "extra_trees",
    "hist_gradient_boosting",
    "xgboost",
)

@dataclass(frozen=True)
class ExperimentConfig:
    """Serializable model-evaluation configuration."""

    feature_path: str
    model_name: str = "lightgbm"
    feature_set: str = "all"
    target: str = "reported"
    split_strategy: str = "grouped"
    n_splits: int = 5
    seed: int = 42
    device: str = "auto"
    apply_official_exclusions: bool = False
    max_rows: int | None = None
    output_dir: str = "artifacts/results"
    model_params: dict[str, Any] = field(default_factory=dict)

@dataclass(frozen=True)
class FeatureMatrix:
    """Prepared raw feature matrix before fold-specific imputation."""

    x: np.ndarray
    y: np.ndarray
    groups: np.ndarray
    feature_names: list[str]
    label_distribution: dict[str, int]

def git_commit() -> str | None:
    """Return current git commit hash when available without shelling out."""
    try:
        head_path = Path(".git") / "HEAD"
        head = head_path.read_text(encoding="utf-8").strip()
        if head.startswith("ref: "):
            ref_path = Path(".git") / head.split(" ", 1)[1]
            return ref_path.read_text(encoding="utf-8").strip()
        return head
    except Exception:
        return None

def file_sha256(path: str | Path) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def select_feature_columns(frame: pl.DataFrame, feature_set: FeatureSet = "all") -> list[str]:
    """Return candidate numeric feature columns with no questionnaire leakage."""
    leakage = {"SubjectID", "Trial", *SELF_REPORT_COLUMNS}
    if feature_set == "ppg":
        return [c for c in frame.columns if c.startswith("Band_PPG_")]
    if feature_set != "all":
        raise ValueError(f"Unsupported feature_set={feature_set!r}; use 'all' or 'ppg'.")
    out: list[str] = []
    for col, dtype in zip(frame.columns, frame.dtypes, strict=True):
        if col in leakage:
            continue
        if dtype.is_numeric():
            out.append(col)
    return out

def load_feature_matrix(
    feature_path: str | Path,
    *,
    feature_set: FeatureSet,
    target: str,
    apply_official_exclusions: bool = False,
    max_rows: int | None = None,
) -> FeatureMatrix:
    """Load feature table and resolve X/y/groups arrays."""
    target_spec = resolve_target(target)
    frame = read_feature_table(feature_path)
    if apply_official_exclusions:
        frame = filter_official_valid_trials(frame)
    if target_spec.column not in frame.columns:
        raise ValueError(f"{target_spec.column} is missing from {feature_path}; rebuild features with target columns.")
    if "SubjectID" not in frame.columns:
        raise ValueError("SubjectID column is required for participant-grouped splits")
    feature_names = select_feature_columns(frame, feature_set=feature_set)
    if not feature_names:
        raise ValueError(f"No feature columns found for feature_set={feature_set!r}")

    frame = frame.with_columns(pl.col(target_spec.column).cast(pl.Int64, strict=False))
    frame = frame.filter(pl.col(target_spec.column).is_in(EMOTION_IDS))
    if max_rows is not None:
        frame = frame.head(max_rows)
    feature_frame = frame.select([pl.col(c).cast(pl.Float64, strict=False).alias(c) for c in feature_names])
    x = feature_frame.to_numpy().astype(float, copy=False)
    y = frame[target_spec.column].to_numpy().astype(int, copy=False)
    groups = frame["SubjectID"].cast(pl.Utf8).to_numpy()
    labels, counts = np.unique(y, return_counts=True)
    distribution = {str(int(label)): int(count) for label, count in zip(labels, counts, strict=True)}
    return FeatureMatrix(x=x, y=y, groups=groups, feature_names=feature_names, label_distribution=distribution)

def _fold_transform(
    x_train: np.ndarray,
    x_test: np.ndarray,
    feature_names: Sequence[str],
    *,
    max_nan_ratio: float = 0.5,
) -> tuple[np.ndarray, np.ndarray, list[str], dict[str, Any]]:
    """Drop unstable columns and median-impute using training-fold statistics only."""
    x_tr = np.asarray(x_train, dtype=float).copy()
    x_te = np.asarray(x_test, dtype=float).copy()
    x_tr[~np.isfinite(x_tr)] = np.nan
    x_te[~np.isfinite(x_te)] = np.nan
    nan_ratio = np.mean(np.isnan(x_tr), axis=0)
    keep = nan_ratio < max_nan_ratio
    # Drop constants after considering finite training values.
    for idx in range(x_tr.shape[1]):
        if not keep[idx]:
            continue
        finite = x_tr[:, idx][np.isfinite(x_tr[:, idx])]
        if finite.size == 0 or np.unique(finite).size <= 1:
            keep[idx] = False
    if not np.any(keep):
        raise ValueError("All feature columns were dropped by fold preprocessing")
    x_tr = x_tr[:, keep]
    x_te = x_te[:, keep]
    selected_names = [name for name, flag in zip(feature_names, keep, strict=True) if flag]
    medians = np.nanmedian(x_tr, axis=0)
    medians[~np.isfinite(medians)] = 0.0
    train_nan = np.isnan(x_tr)
    test_nan = np.isnan(x_te)
    x_tr[train_nan] = np.take(medians, np.where(train_nan)[1])
    x_te[test_nan] = np.take(medians, np.where(test_nan)[1])
    info = {
        "input_features": len(feature_names),
        "selected_features": len(selected_names),
        "dropped_features": int(len(feature_names) - len(selected_names)),
    }
    return x_tr, x_te, selected_names, info

def make_splitter(strategy: SplitStrategy, n_splits: int, seed: int):
    """Create sklearn splitter."""
    if strategy == "grouped":
        from sklearn.model_selection import StratifiedGroupKFold

        return StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    if strategy == "stratified":
        from sklearn.model_selection import StratifiedKFold

        return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    raise ValueError(f"Unsupported split_strategy={strategy!r}")

def split_indices(matrix: FeatureMatrix, strategy: SplitStrategy, n_splits: int, seed: int):
    splitter = make_splitter(strategy, n_splits=n_splits, seed=seed)
    if strategy == "grouped":
        yield from splitter.split(matrix.x, matrix.y, matrix.groups)
    else:
        yield from splitter.split(matrix.x, matrix.y)

### 3-a. Matrix diagnostics

In [ ]:
base_matrix = load_feature_matrix(FEATURE_PPG, feature_set="ppg", target="reported")
rich_matrix = load_feature_matrix(FEATURE_PPG_RICH, feature_set="ppg", target="reported")
display(pd.DataFrame([
    {"variant": "ppg", "rows": base_matrix.y.size, "groups": len(np.unique(base_matrix.groups)), "features": len(base_matrix.feature_names), "labels": base_matrix.label_distribution},
    {"variant": "ppg_rich", "rows": rich_matrix.y.size, "groups": len(np.unique(rich_matrix.groups)), "features": len(rich_matrix.feature_names), "labels": rich_matrix.label_distribution},
]))

## 4. Modeling harness: estimator factory and CV execution

Source: `modeling.py`. 이 셀에서 대안모델 estimator와 CV loop가 직접 정의됩니다.

In [ ]:
DeviceRequest = Literal["auto", "cuda", "cpu"]

@dataclass(frozen=True)
class BackendSelection:
    """Resolved backend metadata for reproducible experiment logs."""

    requested: str
    selected: str
    backend: str
    gpu_name: str | None = None
    fallback_reason: str | None = None

    def to_dict(self) -> dict[str, str | None]:
        return asdict(self)

def select_torch_device(requested: DeviceRequest = "auto") -> BackendSelection:
    """Resolve a PyTorch-style device without requiring torch at import time."""
    if requested == "cpu":
        return BackendSelection(requested=requested, selected="cpu", backend="torch")
    try:
        import torch
    except Exception as exc:  # pragma: no cover - depends on optional install
        if requested == "cuda":
            return BackendSelection(
                requested=requested,
                selected="cpu",
                backend="torch",
                fallback_reason=f"torch import failed: {exc}",
            )
        return BackendSelection(
            requested=requested,
            selected="cpu",
            backend="torch",
            fallback_reason="torch is not installed",
        )

    if torch.cuda.is_available():
        idx = torch.cuda.current_device()
        return BackendSelection(
            requested=requested,
            selected="cuda",
            backend="torch",
            gpu_name=torch.cuda.get_device_name(idx),
        )
    return BackendSelection(
        requested=requested,
        selected="cpu",
        backend="torch",
        fallback_reason="torch.cuda.is_available() is false",
    )

def _probe_lightgbm_device(device_type: str) -> tuple[bool, str | None]:
    """Return whether this LightGBM build can train with a requested device_type."""
    try:
        import lightgbm as lgb
        import numpy as np
    except Exception as exc:  # pragma: no cover - optional dependency
        return False, f"lightgbm import failed: {exc}"

    x = np.array(
        [
            [0.0, 0.0],
            [0.0, 1.0],
            [1.0, 0.0],
            [1.0, 1.0],
            [0.2, 0.1],
            [0.8, 0.9],
        ],
        dtype=float,
    )
    y = np.array([0, 0, 1, 1, 0, 1], dtype=int)
    params = {
        "objective": "multiclass",
        "num_class": 2,
        "metric": "multi_logloss",
        "verbosity": -1,
        "device_type": device_type,
        "num_threads": 1,
        "force_col_wise": True,
    }
    try:
        lgb.train(params, lgb.Dataset(x, label=y), num_boost_round=1)
    except Exception as exc:  # pragma: no cover - depends on local build
        return False, str(exc).splitlines()[0]
    return True, None

def select_lightgbm_device(requested: DeviceRequest = "auto") -> BackendSelection:
    """Resolve LightGBM device_type with CUDA/GPU probing and CPU fallback."""
    if requested == "cpu":
        return BackendSelection(requested=requested, selected="cpu", backend="lightgbm")

    failures: list[str] = []
    for candidate in ("cuda", "gpu"):
        ok, reason = _probe_lightgbm_device(candidate)
        if ok:
            return BackendSelection(requested=requested, selected=candidate, backend="lightgbm")
        failures.append(f"{candidate}: {reason}")

    return BackendSelection(
        requested=requested,
        selected="cpu",
        backend="lightgbm",
        fallback_reason="; ".join(failures),
    )

def select_xgboost_device(requested: DeviceRequest = "auto") -> BackendSelection:
    """Resolve XGBoost device string. XGBoost itself will still validate at fit time."""
    if requested == "cpu":
        return BackendSelection(requested=requested, selected="cpu", backend="xgboost")
    torch_selection = select_torch_device(requested="auto")
    if torch_selection.selected == "cuda":
        return BackendSelection(
            requested=requested,
            selected="cuda",
            backend="xgboost",
            gpu_name=torch_selection.gpu_name,
        )
    return BackendSelection(
        requested=requested,
        selected="cpu",
        backend="xgboost",
        fallback_reason=torch_selection.fallback_reason or "CUDA was not detected",
    )

def _backend_for_model(model_name: str, requested_device: DeviceRequest) -> BackendSelection:
    if model_name == "lightgbm":
        return select_lightgbm_device(requested_device)
    if model_name == "xgboost":
        return select_xgboost_device(requested_device)
    return BackendSelection(requested=requested_device, selected="cpu", backend=model_name)

def create_estimator(
    model_name: ModelName,
    *,
    seed: int,
    device_selection: BackendSelection,
    params: dict[str, Any] | None = None,
):
    """Instantiate a supported classifier."""
    params = dict(params or {})
    if model_name == "lightgbm":
        from lightgbm import LGBMClassifier

        base = {
            "n_estimators": 400,
            "learning_rate": 0.05,
            "max_depth": -1,
            "num_leaves": 63,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_alpha": 0.0,
            "reg_lambda": 0.1,
            "min_child_samples": 20,
            "min_split_gain": 0.0,
            "class_weight": "balanced",
            "random_state": seed,
            "n_jobs": 1,
            "deterministic": True,
            "force_row_wise": True,
            "verbosity": -1,
            "device_type": device_selection.selected,
        }
        base.update(params)
        return LGBMClassifier(**base)

    if model_name == "logistic_regression":
        from sklearn.linear_model import LogisticRegression
        from sklearn.pipeline import make_pipeline
        from sklearn.preprocessing import StandardScaler

        base = {"max_iter": 2000, "class_weight": "balanced", "random_state": seed}
        base.update(params)
        return make_pipeline(StandardScaler(), LogisticRegression(**base))

    if model_name == "svc_rbf":
        from sklearn.pipeline import make_pipeline
        from sklearn.preprocessing import StandardScaler
        from sklearn.svm import SVC

        base = {"C": 3.0, "gamma": "scale", "class_weight": "balanced", "probability": True, "random_state": seed}
        base.update(params)
        return make_pipeline(StandardScaler(), SVC(**base))

    if model_name == "random_forest":
        from sklearn.ensemble import RandomForestClassifier

        base = {
            "n_estimators": 500,
            "max_features": "sqrt",
            "min_samples_leaf": 1,
            "class_weight": "balanced_subsample",
            "random_state": seed,
            "n_jobs": -1,
        }
        base.update(params)
        return RandomForestClassifier(**base)

    if model_name == "extra_trees":
        from sklearn.ensemble import ExtraTreesClassifier

        base = {
            "n_estimators": 700,
            "max_features": "sqrt",
            "min_samples_leaf": 1,
            "class_weight": "balanced",
            "random_state": seed,
            "n_jobs": -1,
        }
        base.update(params)
        return ExtraTreesClassifier(**base)

    if model_name == "hist_gradient_boosting":
        from sklearn.ensemble import HistGradientBoostingClassifier

        base = {"learning_rate": 0.05, "max_iter": 300, "l2_regularization": 0.01, "random_state": seed}
        base.update(params)
        return HistGradientBoostingClassifier(**base)

    if model_name == "xgboost":
        from xgboost import XGBClassifier

        base = {
            "n_estimators": 400,
            "learning_rate": 0.05,
            "max_depth": 4,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_lambda": 1.0,
            "objective": "multi:softprob",
            "eval_metric": "mlogloss",
            "random_state": seed,
            "n_jobs": 1,
            "device": device_selection.selected,
        }
        base.update(params)
        return XGBClassifier(**base)

    raise ValueError(f"Unsupported model_name={model_name!r}")

def _predict_scores(model: Any, x: np.ndarray, labels: Sequence[int] = EMOTION_IDS) -> tuple[np.ndarray, np.ndarray]:
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="X does not have valid feature names.*")
        pred = model.predict(x)
    pred_arr = np.asarray(pred, dtype=int)
    if hasattr(model, "predict_proba"):
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="X does not have valid feature names.*")
            proba = np.asarray(model.predict_proba(x), dtype=float)
        classes = getattr(model, "classes_", labels)
        if not hasattr(model, "classes_") and hasattr(model, "named_steps"):
            # sklearn Pipeline exposes classes_ on the final estimator only in some versions.
            classes = getattr(model.steps[-1][1], "classes_", labels)
        scores = align_proba_columns(classes, proba, labels=labels)
    else:
        scores = None
    if scores is None:
        scores = align_proba_columns(labels, np.eye(len(labels))[np.searchsorted(labels, pred_arr)], labels=labels)
    return pred_arr, scores

def run_cross_validated_experiment(config: ExperimentConfig) -> dict[str, Any]:
    """Run a CV experiment and return a JSON-serializable result bundle."""
    start = time.time()
    if config.model_name not in MODEL_NAMES:
        raise ValueError(f"Unsupported model {config.model_name!r}; choices={MODEL_NAMES}")
    matrix = load_feature_matrix(
        config.feature_path,
        feature_set=config.feature_set,  # type: ignore[arg-type]
        target=config.target,
        apply_official_exclusions=config.apply_official_exclusions,
        max_rows=config.max_rows,
    )
    if len(np.unique(matrix.y)) < 2:
        raise ValueError("Need at least two target classes for classification")
    backend = _backend_for_model(config.model_name, config.device)  # type: ignore[arg-type]
    y_true_all: list[int] = []
    y_pred_all: list[int] = []
    score_all: list[np.ndarray] = []
    fold_summaries: list[dict[str, Any]] = []

    for fold_idx, (train_idx, test_idx) in enumerate(
        split_indices(matrix, config.split_strategy, config.n_splits, config.seed), start=1
    ):
        x_train, x_test, selected_names, transform_info = _fold_transform(
            matrix.x[train_idx], matrix.x[test_idx], matrix.feature_names
        )
        y_train = matrix.y[train_idx]
        y_test = matrix.y[test_idx]
        model = create_estimator(
            config.model_name,  # type: ignore[arg-type]
            seed=config.seed + fold_idx,
            device_selection=backend,
            params=config.model_params,
        )
        fit_start = time.time()
        if config.model_name == "lightgbm":
            try:
                from lightgbm.callback import early_stopping, log_evaluation

                model.fit(
                    x_train,
                    y_train,
                    eval_set=[(x_test, y_test)],
                    eval_metric="multi_logloss",
                    callbacks=[early_stopping(stopping_rounds=50, verbose=False), log_evaluation(period=0)],
                )
            except TypeError:
                model.fit(x_train, y_train)
        else:
            model.fit(x_train, y_train)
        y_pred, scores = _predict_scores(model, x_test)
        fold_metrics = compute_classification_metrics(y_test, y_pred, scores).to_dict()
        y_true_all.extend(y_test.tolist())
        y_pred_all.extend(y_pred.tolist())
        score_all.append(scores)
        fold_summaries.append(
            {
                "fold": fold_idx,
                "train_rows": int(train_idx.size),
                "test_rows": int(test_idx.size),
                "train_groups": int(np.unique(matrix.groups[train_idx]).size),
                "test_groups": int(np.unique(matrix.groups[test_idx]).size),
                "fit_seconds": round(time.time() - fit_start, 3),
                **transform_info,
                "metrics": fold_metrics,
            }
        )
        print(f"[fold {fold_idx}] {metrics_summary_line(fold_metrics)}", flush=True)

    all_scores = np.vstack(score_all)
    overall_metrics = compute_classification_metrics(y_true_all, y_pred_all, all_scores).to_dict()
    elapsed = time.time() - start
    result = {
        "schema_version": 1,
        "created_at_unix": int(time.time()),
        "elapsed_seconds": round(elapsed, 3),
        "config": asdict(config),
        "environment": {
            "llamac_research_version": __version__,
            "python": platform.python_version(),
            "platform": platform.platform(),
            "git_commit": git_commit(),
            "feature_file_sha256": file_sha256(config.feature_path),
        },
        "backend": backend.to_dict(),
        "data": {
            "rows": int(matrix.y.size),
            "candidate_features": int(len(matrix.feature_names)),
            "label_distribution": matrix.label_distribution,
            "groups": int(np.unique(matrix.groups).size),
        },
        "folds": fold_summaries,
        "metrics": overall_metrics,
    }
    return result

def default_result_path(config: ExperimentConfig, result: dict[str, Any]) -> Path:
    """Build deterministic-ish output path from config and timestamp."""
    stamp = time.strftime("%Y%m%d-%H%M%S", time.localtime(result["created_at_unix"]))
    official = "official" if config.apply_official_exclusions else "allsubjects"
    name = f"{stamp}_{config.model_name}_{config.feature_set}_{config.target}_{config.split_strategy}_{official}.json"
    return Path(config.output_dir) / name

def save_experiment_result(result: dict[str, Any], output_path: str | Path) -> None:
    out = Path(output_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(result, indent=2, sort_keys=True), encoding="utf-8")

### 4-a. Backend probe for the current `DEVICE` setting

이 셀은 대안모델 노트북에서 `DEVICE`가 실제로 어떤 backend/device로 해석되는지 바로 보여줍니다.
LightGBM은 설치된 build가 CUDA/GPU를 지원하는지 작은 probe 학습으로 확인한 뒤 CPU fallback 이유를 기록합니다.


In [ ]:
print(f"Resolving model backends for DEVICE={DEVICE!r} ...")
lightgbm_backend_probe = select_lightgbm_device(DEVICE)
xgboost_backend_probe = select_xgboost_device(DEVICE)
torch_backend_probe = select_torch_device(DEVICE)

def print_backend_selection(name: str, selection: BackendSelection) -> None:
    print(f"[{name}]")
    print(f"  requested: {selection.requested}")
    print(f"  selected: {selection.selected}")
    print(f"  backend: {selection.backend}")
    if selection.gpu_name:
        print(f"  gpu: {selection.gpu_name}")
    if selection.fallback_reason:
        print(f"  fallback_reason: {selection.fallback_reason}")

print_backend_selection("LightGBM", lightgbm_backend_probe)
print_backend_selection("XGBoost", xgboost_backend_probe)
print_backend_selection("Torch", torch_backend_probe)

display(pd.DataFrame([
    {"model_family": "lightgbm", **lightgbm_backend_probe.to_dict()},
    {"model_family": "xgboost", **xgboost_backend_probe.to_dict()},
    {"model_family": "torch", **torch_backend_probe.to_dict()},
]))


## 5. Optuna tuning implementation

Source: `tuning.py`. search space, storage, trial objective, locked-best re-evaluation이 모두 보이는 셀입니다.

In [ ]:
@dataclass(frozen=True)
class TuningConfig:
    """Serializable Optuna tuning configuration."""

    feature_path: str
    models: list[str]
    feature_set: str = "ppg"
    target: str = "reported"
    split_strategy: str = "grouped"
    n_splits: int = 5
    seed: int = 42
    device: str = "auto"
    apply_official_exclusions: bool = False
    n_trials: int = 20
    timeout_seconds: int | None = None
    metric: str = "macro_f1"
    output_dir: str = "artifacts/optuna"
    storage: str | None = None
    study_prefix: str = "llamac"
    max_rows: int | None = None

def sample_model_params(trial: Any, model_name: str) -> dict[str, Any]:
    """Return model-specific Optuna search-space parameters."""
    if model_name == "lightgbm":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 150, 900),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 15, 127),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 80),
            "subsample": trial.suggest_float("subsample", 0.55, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.55, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        }
    if model_name == "logistic_regression":
        return {
            "C": trial.suggest_float("C", 1e-3, 100.0, log=True),
            "solver": "lbfgs",
        }
    if model_name == "svc_rbf":
        return {
            "C": trial.suggest_float("C", 0.1, 100.0, log=True),
            "gamma": trial.suggest_float("gamma", 1e-4, 1.0, log=True),
        }
    if model_name == "random_forest":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 200, 900),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 8),
            "max_depth": trial.suggest_categorical("max_depth", [None, 4, 6, 8, 12, 16]),
        }
    if model_name == "extra_trees":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 300, 1000),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 8),
            "max_depth": trial.suggest_categorical("max_depth", [None, 4, 6, 8, 12, 16]),
        }
    if model_name == "hist_gradient_boosting":
        return {
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "max_iter": trial.suggest_int("max_iter", 100, 700),
            "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 7, 63),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 80),
            "l2_regularization": trial.suggest_float("l2_regularization", 1e-8, 10.0, log=True),
        }
    if model_name == "xgboost":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 150, 800),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "max_depth": trial.suggest_int("max_depth", 2, 8),
            "subsample": trial.suggest_float("subsample", 0.55, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.55, 1.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-6, 20.0, log=True),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        }
    raise ValueError(f"Unsupported model_name={model_name!r}")

def _study_storage(config: TuningConfig, model_name: str) -> str | None:
    if config.storage:
        return config.storage
    out = Path(config.output_dir)
    out.mkdir(parents=True, exist_ok=True)
    return f"sqlite:///{out / f'{config.study_prefix}_{model_name}_{config.feature_set}_{config.target}.db'}"

def run_tuning(config: TuningConfig) -> dict[str, Any]:
    """Run Optuna studies for one or more model families."""
    import optuna

    invalid = [model for model in config.models if model not in MODEL_NAMES]
    if invalid:
        raise ValueError(f"Unsupported models {invalid}; choices={MODEL_NAMES}")

    started = time.time()
    summaries: list[dict[str, Any]] = []
    final_results: list[dict[str, Any]] = []
    for model_name in config.models:
        storage = _study_storage(config, model_name)
        study_name = f"{config.study_prefix}_{model_name}_{config.feature_set}_{config.target}_{config.split_strategy}"
        study = optuna.create_study(
            study_name=study_name,
            storage=storage,
            direction="maximize",
            load_if_exists=True,
            sampler=optuna.samplers.TPESampler(seed=config.seed),
        )

        def objective(trial: Any) -> float:
            params = sample_model_params(trial, model_name)
            exp = ExperimentConfig(
                feature_path=config.feature_path,
                model_name=model_name,
                feature_set=config.feature_set,
                target=config.target,
                split_strategy=config.split_strategy,
                n_splits=config.n_splits,
                seed=config.seed,
                device=config.device,
                apply_official_exclusions=config.apply_official_exclusions,
                max_rows=config.max_rows,
                output_dir=str(Path(config.output_dir) / "trial-results"),
                model_params=params,
            )
            result = run_cross_validated_experiment(exp)
            value = float(result["metrics"][config.metric])
            trial.set_user_attr("metrics", result["metrics"])
            trial.set_user_attr("backend", result["backend"])
            trial.set_user_attr("data", result["data"])
            return value

        study.optimize(objective, n_trials=config.n_trials, timeout=config.timeout_seconds, gc_after_trial=True)
        best_params = dict(study.best_trial.params)
        locked = ExperimentConfig(
            feature_path=config.feature_path,
            model_name=model_name,
            feature_set=config.feature_set,
            target=config.target,
            split_strategy=config.split_strategy,
            n_splits=config.n_splits,
            seed=config.seed,
            device=config.device,
            apply_official_exclusions=config.apply_official_exclusions,
            max_rows=config.max_rows,
            output_dir=str(Path(config.output_dir) / "locked-results"),
            model_params=best_params,
        )
        locked_result = run_cross_validated_experiment(locked)
        locked_path = Path(config.output_dir) / "locked-results" / (
            f"{model_name}_{config.feature_set}_{config.target}_{config.split_strategy}_best.json"
        )
        save_experiment_result(locked_result, locked_path)
        summary = {
            "model_name": model_name,
            "study_name": study.study_name,
            "storage": storage,
            "n_trials_total": len(study.trials),
            "best_trial_number": study.best_trial.number,
            "best_value": float(study.best_value),
            "metric": config.metric,
            "best_params": best_params,
            "locked_result_path": str(locked_path),
            "locked_metrics": locked_result["metrics"],
        }
        print(
            f"[optuna:{model_name}] best_{config.metric}={study.best_value:.4f} "
            f"locked {metrics_summary_line(locked_result['metrics'])}",
            flush=True,
        )
        summaries.append(summary)
        final_results.append(locked_result)

    out = {
        "schema_version": 1,
        "created_at_unix": int(time.time()),
        "elapsed_seconds": round(time.time() - started, 3),
        "config": asdict(config),
        "studies": summaries,
    }
    out_dir = Path(config.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    summary_path = out_dir / f"{config.study_prefix}_{config.feature_set}_{config.target}_summary.json"
    summary_path.write_text(json.dumps(out, indent=2, sort_keys=True), encoding="utf-8")
    out["summary_path"] = str(summary_path)
    print(f"[optuna:summary] {summary_path}", flush=True)
    return out

## 6. Run or load base PPG Optuna studies

`RERUN_TUNING=True`로 바꾸면 search를 다시 실행합니다. 기본값은 locked result가 있으면 읽습니다.

In [ ]:
BASE_MODELS = ["lightgbm", "extra_trees", "hist_gradient_boosting", "random_forest", "logistic_regression"]
BASE_EXPECTED = [OPTUNA_BASE / "locked-results" / f"{model}_ppg_reported_grouped_best.json" for model in BASE_MODELS]
base_config = TuningConfig(
    feature_path=str(FEATURE_PPG),
    models=BASE_MODELS,
    feature_set="ppg",
    target="reported",
    split_strategy="grouped",
    n_splits=FOLDS,
    seed=SEED,
    device=DEVICE,
    n_trials=BASE_TRIALS,
    metric="macro_f1",
    output_dir=str(OPTUNA_BASE),
)
if all(path.exists() for path in BASE_EXPECTED) and not RERUN_TUNING:
    print("[load] base PPG locked Optuna results exist")
else:
    base_summary = run_tuning(base_config)

## 7. Run or load slower/focused studies: SVC-RBF and rich PPG

In [ ]:
svc_expected = OPTUNA_BASE / "locked-results" / "svc_rbf_ppg_reported_grouped_best.json"
svc_config = TuningConfig(
    feature_path=str(FEATURE_PPG),
    models=["svc_rbf"],
    feature_set="ppg",
    target="reported",
    split_strategy="grouped",
    n_splits=FOLDS,
    seed=SEED,
    device=DEVICE,
    n_trials=SVC_TRIALS,
    metric="macro_f1",
    output_dir=str(OPTUNA_BASE),
)
if svc_expected.exists() and not RERUN_TUNING:
    print("[load] SVC-RBF locked result exists")
else:
    svc_summary = run_tuning(svc_config)

RICH_MODELS = ["lightgbm", "logistic_regression"]
rich_expected = [OPTUNA_RICH / "locked-results" / f"{model}_ppg_reported_grouped_best.json" for model in RICH_MODELS]
rich_config = TuningConfig(
    feature_path=str(FEATURE_PPG_RICH),
    models=RICH_MODELS,
    feature_set="ppg",
    target="reported",
    split_strategy="grouped",
    n_splits=FOLDS,
    seed=SEED,
    device=DEVICE,
    n_trials=RICH_TRIALS,
    metric="macro_f1",
    output_dir=str(OPTUNA_RICH),
)
if all(path.exists() for path in rich_expected) and not RERUN_TUNING:
    print("[load] rich PPG locked results exist")
else:
    rich_summary = run_tuning(rich_config)

## 8. Collect locked results and compare model families

In [ ]:
def load_locked_result(path: Path, variant: str) -> dict:
    result = json.loads(path.read_text())
    cfg, data, m = result["config"], result["data"], result["metrics"]
    return {
        "path": str(path),
        "variant": variant,
        "model": cfg["model_name"],
        "target": cfg["target"],
        "split": cfg["split_strategy"],
        "rows": data["rows"],
        "features": data["candidate_features"],
        "top1": m["top1_accuracy"],
        "top2": m["top2_accuracy"],
        "top3": m["top3_accuracy"],
        "macro_f1": m["macro_f1"],
        "weighted_f1": m["weighted_f1"],
        "balanced_acc": m["balanced_accuracy"],
        "kappa": m["cohen_kappa"],
        "best_params": cfg["model_params"],
    }

locked_rows = []
for path in sorted((OPTUNA_BASE / "locked-results").glob("*_best.json")):
    locked_rows.append(load_locked_result(path, "ppg"))
for path in sorted((OPTUNA_RICH / "locked-results").glob("*_best.json")):
    locked_rows.append(load_locked_result(path, "ppg_rich"))

locked_table = pd.DataFrame(locked_rows).sort_values("macro_f1", ascending=False)
metric_cols = ["top1", "top2", "top3", "macro_f1", "weighted_f1", "balanced_acc", "kappa"]
display(locked_table.drop(columns=["path", "best_params"]).style.format({c: "{:.4f}" for c in metric_cols}))

## 9. Best model details and confusion matrix

In [ ]:
best = locked_table.iloc[0]
best_result = json.loads(Path(best["path"]).read_text())
display(Markdown(
    f"Best PPG-only alternative: **{best['variant']} + {best['model']}** "
    f"with macro F1 **{best['macro_f1']:.4f}** and top-1 **{best['top1']:.4f}**."
))
print(json.dumps(best["best_params"], indent=2, sort_keys=True))

cm = np.asarray(best_result["metrics"]["confusion_matrix"])
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Purples")
ax.set_title(f"{best['variant']} + {best['model']}")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_xticks(range(len(EMOTION_LABELS)), EMOTION_LABELS, rotation=45, ha="right")
ax.set_yticks(range(len(EMOTION_LABELS)), EMOTION_LABELS)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## 10. Visual comparison and optional notebook summary artifact

In [ ]:
plot_df = locked_table.sort_values("macro_f1")
labels_for_plot = plot_df["variant"] + " + " + plot_df["model"]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(labels_for_plot, plot_df["macro_f1"])
ax.set_xlabel("Macro F1")
ax.set_title("PPG-only Optuna locked configurations")
for value, label in zip(plot_df["macro_f1"], labels_for_plot):
    ax.text(value + 0.002, label, f"{value:.3f}", va="center")
plt.tight_layout()
plt.show()

summary_path = RESULTS / "notebook_ppg_alternative_summary.csv"
locked_table.drop(columns=["best_params"]).to_csv(summary_path, index=False)
print(summary_path)